<a href="https://colab.research.google.com/github/mena-04/qml/blob/main/phase7_data_reuploading_qnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports and Configuration

In [ ]:

%pip install -q pennylane torch scikit-learn pandas numpy matplotlib


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
import random
import time
import zipfile

from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split
)
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

SEED = 42
N_QUBITS = 4
N_LAYERS = 2
EPOCHS = 30
LR = 0.03

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

OUTPUT_DIR = Path("phase_7_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)


PyTorch: 2.11.0+cpu
PennyLane: 0.45.1


Shared Breast Cancer Training Data

In [ ]:

from google.colab import files

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

dataset = load_breast_cancer()

X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

with np.load(
    "limited_data/data_n40_seed42.npz"
) as data:
    subset_idx = data["train_idx"].copy()

X_subset = X_raw[subset_idx]
y_subset = y_all[subset_idx]

print("Training subset:", X_subset.shape)
print("Malignant:", np.sum(y_subset == 1))
print("Benign:", np.sum(y_subset == 0))


Saving qbreast_limited_data.zip to qbreast_limited_data.zip
Training subset: (40, 30)
Malignant: 15
Benign: 25


Leakage-Safe Preprocessing

In [ ]:

X_train_raw, X_val_raw, y_train, y_val = (
    train_test_split(
        X_subset,
        y_subset,
        test_size=0.25,
        stratify=y_subset,
        random_state=SEED
    )
)

def make_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])

preprocessor = make_preprocessor()

X_train = preprocessor.fit_transform(
    X_train_raw
)

X_val = preprocessor.transform(
    X_val_raw
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)


Training: (30, 4)
Validation: (10, 4)


Standard vs Data-Reuploading Circuits

In [ ]:

class HybridQuantumClassifier(nn.Module):

    def __init__(self, reupload=False):
        super().__init__()

        self.reupload = reupload

        dev = qml.device(
            "default.qubit",
            wires=N_QUBITS
        )

        @qml.qnode(
            dev,
            interface="torch",
            diff_method="backprop"
        )
        def circuit(inputs, weights):

            # Initial data encoding
            for i in range(N_QUBITS):
                qml.RY(inputs[i], wires=i)

            for layer in range(N_LAYERS):

                # Trainable quantum rotations
                for i in range(N_QUBITS):
                    qml.RY(
                        weights[layer, i, 0],
                        wires=i
                    )
                    qml.RZ(
                        weights[layer, i, 1],
                        wires=i
                    )

                # Entangling CNOT chain
                for i in range(N_QUBITS - 1):
                    qml.CNOT(
                        wires=[i, i + 1]
                    )

                # Re-upload inputs between layers
                if reupload and layer < N_LAYERS - 1:
                    for i in range(N_QUBITS):
                        qml.RY(inputs[i], wires=i)

            return [
                qml.expval(qml.PauliZ(i))
                for i in range(N_QUBITS)
            ]

        self.quantum = qml.qnn.TorchLayer(
            circuit,
            {
                "weights": (
                    N_LAYERS,
                    N_QUBITS,
                    2
                )
            }
        )

        self.output = nn.Linear(
            N_QUBITS, 1
        )

    def forward(self, x):

        outputs = torch.stack([
            self.quantum(sample)
            for sample in x
        ])

        return self.output(
            outputs
        ).squeeze(-1)


Classical Neural Network Baseline

In [ ]:

class ClassicalMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 4),
            nn.Tanh(),
            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


Architecture Sanity Checks

In [ ]:

models = {
    "Classical MLP": ClassicalMLP(),
    "QNN-C": HybridQuantumClassifier(
        reupload=False
    ),
    "Reuploading QNN": HybridQuantumClassifier(
        reupload=True
    )
}

for name, model in models.items():

    n_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print(name, "parameters:", n_params)

# Confirm reuploading changes the circuit output
torch.manual_seed(SEED)

standard = HybridQuantumClassifier(
    reupload=False
).double()

torch.manual_seed(SEED)

reupload = HybridQuantumClassifier(
    reupload=True
).double()

sample = torch.tensor(
    X_train[:2],
    dtype=torch.float64
)

with torch.no_grad():
    out_standard = standard(sample)
    out_reupload = reupload(sample)

difference = torch.max(
    torch.abs(
        out_standard - out_reupload
    )
).item()

print("Output difference:", difference)

assert difference > 1e-8

# Verify quantum gradients
reupload.train()

loss = nn.BCEWithLogitsLoss()(
    reupload(sample),
    torch.tensor(
        y_train[:2],
        dtype=torch.float64
    )
)

loss.backward()

quantum_grad = (
    reupload.quantum.weights.grad.norm().item()
)

print("Quantum gradient norm:", quantum_grad)

assert np.isfinite(quantum_grad)
assert quantum_grad > 1e-10

print("Reuploading QNN sanity checks passed.")


Classical MLP parameters: 25
QNN-C parameters: 21
Reuploading QNN parameters: 21
Output difference: 1.1379020840948684
Quantum gradient norm: 0.24085550090813396
Reuploading QNN sanity checks passed.


Train and Evaluate Neural Models

In [ ]:

def train_and_evaluate(
    model_name,
    Xtr, ytr,
    Xval, yval,
    seed=42,
    epochs=30,
    lr=0.03
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if model_name == "Classical MLP":
        model = ClassicalMLP().double()

    elif model_name == "QNN-C":
        model = HybridQuantumClassifier(
            reupload=False
        ).double()

    elif model_name == "Reuploading QNN":
        model = HybridQuantumClassifier(
            reupload=True
        ).double()

    else:
        raise ValueError(model_name)

    Xtr_t = torch.tensor(
        Xtr, dtype=torch.float64
    )

    Xval_t = torch.tensor(
        Xval, dtype=torch.float64
    )

    ytr_t = torch.tensor(
        ytr, dtype=torch.float64
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.BCEWithLogitsLoss()

    start = time.perf_counter()

    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)

        loss.backward()
        optimizer.step()

    elapsed = time.perf_counter() - start

    model.eval()

    with torch.no_grad():

        val_logits = model(Xval_t)

        probs = torch.sigmoid(
            val_logits
        ).cpu().numpy()

        train_loss = criterion(
            model(Xtr_t),
            ytr_t
        ).item()

    pred = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yval,
        pred,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            yval, probs
        ),
        "recall": recall_score(
            yval, pred,
            zero_division=0
        ),
        "precision": precision_score(
            yval, pred,
            zero_division=0
        ),
        "f1": f1_score(
            yval, pred,
            zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "final_train_loss": train_loss,
        "training_seconds": elapsed
    }


Initial Reuploading QNN Benchmark

In [ ]:

pilot_records = []

for model_name in [
    "Classical MLP",
    "QNN-C",
    "Reuploading QNN"
]:

    result = train_and_evaluate(
        model_name,
        X_train,
        y_train,
        X_val,
        y_val,
        seed=SEED,
        epochs=EPOCHS,
        lr=LR
    )

    pilot_records.append({
        "model": model_name,
        **result
    })

    print(
        model_name,
        "AUC:",
        round(result["roc_auc"], 4),
        "Recall:",
        round(result["recall"], 4)
    )

pilot_df = pd.DataFrame(pilot_records)

display(pilot_df.round(4))

pilot_df.to_csv(
    OUTPUT_DIR / "phase7_pilot.csv",
    index=False
)


Classical MLP AUC: 1.0 Recall: 0.5
QNN-C AUC: 0.9167 Recall: 0.5
Reuploading QNN AUC: 0.8333 Recall: 0.0


,model,roc_auc,recall,precision,f1,specificity,false_negatives,false_positives,final_train_loss,training_seconds
0,Classical MLP,1.0000,0.5,1.0,0.6667,1.0,2,0,0.2958,0.0497
1,QNN-C,0.9167,0.5,1.0,0.6667,1.0,2,0,0.2283,14.3577
2,Reuploading QNN,0.8333,0.0,0.0,0.0000,1.0,4,0,0.4551,16.7301


Configure repeated evaluation

In [ ]:

P710_SIZES = [40, 80, 160]
P710_SEEDS = [42, 123, 2026]

P710_MODELS = [
    "Classical MLP",
    "QNN-C",
    "Reuploading QNN"
]

P710_EPOCHS = 30
P710_LR = 0.03

p710_records = []

print("Phase 7.10 ready.")


Phase 7.10 ready.


Run repeated cross-validation

In [ ]:

p710_records = []

for n_samples in P710_SIZES:
    for seed in P710_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            prep = make_preprocessor()

            Xtr = prep.fit_transform(
                X_subset[tr]
            )

            Xval = prep.transform(
                X_subset[val]
            )

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P710_MODELS:

                result = train_and_evaluate(
                    model_name,
                    Xtr, ytr,
                    Xval, yval,
                    seed=seed + fold,
                    epochs=P710_EPOCHS,
                    lr=P710_LR
                )

                p710_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    **result
                })

                print(
                    f"n={n_samples}, seed={seed}, "
                    f"fold={fold}, {model_name}: "
                    f"AUC={result['roc_auc']:.4f}, "
                    f"Recall={result['recall']:.4f}"
                )

        # Save after every seed
        pd.DataFrame(p710_records).to_csv(
            OUTPUT_DIR / "phase710_raw.csv",
            index=False
        )

p710_df = pd.DataFrame(p710_records)

print("Phase 7.10 complete.")


n=40, seed=42, fold=0, Classical MLP: AUC=1.0000, Recall=0.8000
n=40, seed=42, fold=0, QNN-C: AUC=0.9778, Recall=0.6000
n=40, seed=42, fold=0, Reuploading QNN: AUC=0.8889, Recall=0.8000
n=40, seed=42, fold=1, Classical MLP: AUC=0.8250, Recall=0.6000
n=40, seed=42, fold=1, QNN-C: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=1, Reuploading QNN: AUC=0.9250, Recall=0.6000
n=40, seed=42, fold=2, Classical MLP: AUC=0.9250, Recall=0.8000
n=40, seed=42, fold=2, QNN-C: AUC=0.9750, Recall=1.0000
n=40, seed=42, fold=2, Reuploading QNN: AUC=0.9500, Recall=1.0000
n=40, seed=123, fold=0, Classical MLP: AUC=0.9556, Recall=0.6000
n=40, seed=123, fold=0, QNN-C: AUC=1.0000, Recall=0.6000
n=40, seed=123, fold=0, Reuploading QNN: AUC=1.0000, Recall=0.6000
n=40, seed=123, fold=1, Classical MLP: AUC=0.8750, Recall=0.6000
n=40, seed=123, fold=1, QNN-C: AUC=0.9000, Recall=0.6000
n=40, seed=123, fold=1, Reuploading QNN: AUC=0.9000, Recall=0.6000
n=40, seed=123, fold=2, Classical MLP: AUC=1.0000, Recall=1.0000

Summarize performance

In [ ]:

p710_per_seed = (
    p710_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        specificity=("specificity", "mean"),
        runtime=("training_seconds", "sum")
    )
    .reset_index()
)

p710_summary = (
    p710_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        specificity_mean=("specificity", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p710_summary.round(4))

p710_summary.to_csv(
    OUTPUT_DIR / "phase710_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,recall_std,precision_mean,f1_mean,specificity_mean,runtime_mean
0,40,Classical MLP,0.9534,0.0425,0.7333,0.0000,0.9722,0.8228,0.9861,0.0901
1,40,QNN-C,0.9630,0.0037,0.7556,0.0385,0.8926,0.7978,0.9321,42.8325
2,40,Reuploading QNN,0.9157,0.0291,0.6222,0.1678,0.8870,0.7010,0.9352,46.0277
3,80,Classical MLP,0.9624,0.0272,0.7667,0.1000,0.9620,0.8352,0.9734,0.0746
4,80,QNN-C,0.9677,0.0141,0.7222,0.0192,0.9277,0.8023,0.9661,79.9842
5,80,Reuploading QNN,0.9621,0.0147,0.6556,0.1644,0.8290,0.7221,0.9657,88.1041
6,160,Classical MLP,0.9862,0.0093,0.7722,0.0694,0.9942,0.8610,0.9966,0.0771
7,160,QNN-C,0.9683,0.0081,0.7389,0.0509,0.9466,0.8264,0.9735,164.1364
8,160,Reuploading QNN,0.9450,0.0092,0.7444,0.0347,0.9235,0.8183,0.9571,179.5981


Compare re-uploading against QNN-C

In [ ]:

paired = p710_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

for metric in ["roc_auc", "recall", "f1"]:
    paired[f"{metric}_difference"] = (
        paired[f"{metric}__Reuploading QNN"]
        - paired[f"{metric}__QNN-C"]
    )

p710_differences = (
    paired
    .groupby("n_samples")
    [[
        "roc_auc_difference",
        "recall_difference",
        "f1_difference"
    ]]
    .agg(["mean", "std"])
)

display(p710_differences.round(4))


roc_auc_difference         recall_difference         f1_difference  \
                        mean     std              mean     std          mean   
n_samples                                                                      
40                   -0.0472  0.0462           -0.1333  0.2000       -0.0968   
80                   -0.0056  0.0304           -0.0667  0.2958       -0.0802   
160                  -0.0232  0.0231            0.0056  0.0527       -0.0080   

                   
              std  
n_samples          
40         0.1633  
80         0.3003  
160        0.0407

Check consistency across folds

In [ ]:

consistency_records = []

for n_samples, group in paired.groupby("n_samples"):

    for metric in ["roc_auc", "recall", "f1"]:

        diff = group[f"{metric}_difference"]

        consistency_records.append({
            "n_samples": n_samples,
            "metric": metric,
            "reupload_wins": int((diff > 0).sum()),
            "ties": int((diff == 0).sum()),
            "qnn_c_wins": int((diff < 0).sum()),
            "mean_difference": diff.mean()
        })

consistency_df = pd.DataFrame(
    consistency_records
)

display(consistency_df.round(4))

consistency_df.to_csv(
    OUTPUT_DIR / "phase710_consistency.csv",
    index=False
)


,n_samples,metric,reupload_wins,ties,qnn_c_wins,mean_difference
0,40,roc_auc,0,3,6,-0.0472
1,40,recall,1,3,5,-0.1333
2,40,f1,2,2,5,-0.0968
3,80,roc_auc,2,1,6,-0.0056
4,80,recall,3,2,4,-0.0667
5,80,f1,3,2,4,-0.0802
6,160,roc_auc,2,0,7,-0.0232
7,160,recall,3,3,3,0.0056
8,160,f1,3,1,5,-0.0080
